# Neural Network

## Data Preparation and Preprocessing

In [1]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# getting rid of " " in TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)# make it numeric, if cannot put NaN(this is what errors="coerce" does)

#we saw that when Total Charges " " tenure=0 meaning new customers so we can
#make TotalCharges=0 for them
df["TotalCharges"] = df["TotalCharges"].fillna(0) #replace NaNs with 0s

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

# drop meaningless feature and target to create X
X = df.drop(columns=["Churn", "customerID"])
# choosing Churn as the target
y = df["Churn"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [2]:
categorical_features = [cname for cname in X.columns if
                    X[cname].dtype == "object"]

# Select numerical columns
numerical_features = [cname for cname in X.columns if 
                X[cname].dtype in ['int64', 'float64']]

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(
        drop="first",
        handle_unknown="ignore"
    ), categorical_features)
])

In [4]:
# Fit the preprocessor only on the training data
X_train_processed = preprocessor.fit_transform(X_train, y_train)

# Apply the already-fitted preprocessor to the test data
X_test_processed = preprocessor.transform(X_test)

In [5]:
# check data shape.. good it is an np.array
print("Training data shape:", X_train_processed.shape)
print("Test data shape:", X_test_processed.shape)
print("Data type:", type(X_train_processed))

Training data shape: (5634, 30)
Test data shape: (1409, 30)
Data type: <class 'numpy.ndarray'>


# Model (Trying out different architectures)

In [6]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input,Dense, Dropout

def build_model(hidden_layers, dropout_rate=0.2):
    model = Sequential()

    model.add(Input(shape=(X_train_processed.shape[1],)))

    for units in hidden_layers:
        model.add(Dense(units, activation="relu"))
        model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

#### Adding Early Stopping System

In [7]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_auc", ## Monitoring val_auc because we are comparing the project models using ROC-AUC.
    patience=10,
    mode="max",
    restore_best_weights=True
)

In [8]:
import numpy as np
import tensorflow as tf

np.random.seed(0)
tf.random.set_seed(0)

## Model 1

In [9]:
model = build_model([32, 16], dropout_rate=0.2)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7280 - auc: 0.7357 - loss: 0.5158 - val_accuracy: 0.7879 - val_auc: 0.8392 - val_loss: 0.4278
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7679 - auc: 0.8050 - loss: 0.4672 - val_accuracy: 0.8137 - val_auc: 0.8493 - val_loss: 0.4130
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7808 - auc: 0.8207 - loss: 0.4510 - val_accuracy: 0.8110 - val_auc: 0.8521 - val_loss: 0.4071
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7859 - auc: 0.8294 - loss: 0.4390 - val_accuracy: 0.8092 - val_auc: 0.8554 - val_loss: 0.4029
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7857 - auc: 0.8309 - loss: 0.4360 - val_accuracy: 0.8154 - val_auc: 0.8558 - val_loss: 0.4029
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7846 - auc: 0.8336 - loss: 0.4337 - val_accuracy: 0.8101 - val_auc: 0.8555 - val_loss: 0.4019
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 

### Model 1- Evaluation

In [10]:
y_prob = model.predict(X_test_processed).ravel()

y_pred = (y_prob >= 0.5).astype(int) # threshold 0.5

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7856635911994322
Precision: 0.5964912280701754
Recall: 0.5543478260869565
F1-score: 0.5746478873239437
ROC-AUC: 0.8188301382449986


## Model 2

In [12]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    restore_best_weights=True
)

In [13]:
model = build_model([16, 8], dropout_rate=0.2)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6732 - auc: 0.5158 - loss: 0.6342 - val_accuracy: 0.7542 - val_auc: 0.8041 - val_loss: 0.4911
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7566 - auc: 0.7668 - loss: 0.4951 - val_accuracy: 0.8004 - val_auc: 0.8397 - val_loss: 0.4185
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7648 - auc: 0.7927 - loss: 0.4725 - val_accuracy: 0.8128 - val_auc: 0.8459 - val_loss: 0.4086
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7748 - auc: 0.8113 - loss: 0.4563 - val_accuracy: 0.8172 - val_auc: 0.8499 - val_loss: 0.4043
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7886 - auc: 0.8181 - loss: 0.4490 - val_accuracy: 0.8146 - val_auc: 0.8524 - val_loss: 0.4019
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7846 - auc: 0.8221 - loss: 0.4457 - val_accuracy: 0.8163 - val_auc: 0.8528 - val_loss: 0.4017
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 

### Model 2- Evaluation

In [14]:
y_prob_2 = model.predict(X_test_processed).ravel()

y_pred_2 = (y_prob_2 >= 0.5).astype(int) # threshold 0.5

print("Accuracy:", accuracy_score(y_test, y_pred_2))
print("Precision:", precision_score(y_test, y_pred_2))
print("Recall:", recall_score(y_test, y_pred_2))
print("F1-score:", f1_score(y_test, y_pred_2))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_2))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Accuracy: 0.7892122072391767
Precision: 0.6126984126984127
Recall: 0.5244565217391305
F1-score: 0.5651537335285505
ROC-AUC: 0.8184438040345821


## Model 3

In [15]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    restore_best_weights=True
)

In [16]:
model = build_model([64, 32], dropout_rate=0.2)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7559 - auc: 0.7719 - loss: 0.4893 - val_accuracy: 0.8199 - val_auc: 0.8501 - val_loss: 0.4009
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7897 - auc: 0.8244 - loss: 0.4427 - val_accuracy: 0.8163 - val_auc: 0.8535 - val_loss: 0.3987
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7943 - auc: 0.8387 - loss: 0.4294 - val_accuracy: 0.8208 - val_auc: 0.8538 - val_loss: 0.3990
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7985 - auc: 0.8420 - loss: 0.4249 - val_accuracy: 0.8208 - val_auc: 0.8549 - val_loss: 0.3988
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8010 - auc: 0.8421 - loss: 0.4245 - val_accuracy: 0.8190 - val_auc: 0.8544 - val_loss: 0.3989
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8010 - auc: 0.8453 - loss: 0.4211 - val_accuracy: 0.8199 - val_auc: 0.8554 - val_loss: 0.3986
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 

### Model 3-Evaluation

In [17]:
y_prob_3 = model.predict(X_test_processed).ravel()

y_pred_3 = (y_prob_3 >= 0.5).astype(int) # threshold 0.5

print("Accuracy:", accuracy_score(y_test, y_pred_3))
print("Precision:", precision_score(y_test, y_pred_3))
print("Recall:", recall_score(y_test, y_pred_3))
print("F1-score:", f1_score(y_test, y_pred_3))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_3))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Accuracy: 0.7856635911994322
Precision: 0.6204379562043796
Recall: 0.46195652173913043
F1-score: 0.5295950155763239
ROC-AUC: 0.8237193542997955


## Model 4

In [18]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    restore_best_weights=True
)

In [19]:
model = build_model([64, 32, 16], dropout_rate=0.2)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7311 - auc: 0.7250 - loss: 0.5265 - val_accuracy: 0.8163 - val_auc: 0.8523 - val_loss: 0.4025
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7812 - auc: 0.8154 - loss: 0.4522 - val_accuracy: 0.8163 - val_auc: 0.8556 - val_loss: 0.3983
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7832 - auc: 0.8300 - loss: 0.4379 - val_accuracy: 0.8225 - val_auc: 0.8563 - val_loss: 0.3980
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7943 - auc: 0.8321 - loss: 0.4372 - val_accuracy: 0.8208 - val_auc: 0.8571 - val_loss: 0.3951
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7910 - auc: 0.8317 - loss: 0.4366 - val_accuracy: 0.8181 - val_auc: 0.8551 - val_loss: 0.3982
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7994 - auc: 0.8358 - loss: 0.4336 - val_accuracy: 0.8172 - val_auc: 0.8551 - val_loss: 0.3979
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 

### Model 4- Evaluation

In [20]:
y_prob_4 = model.predict(X_test_processed).ravel()

y_pred_4 = (y_prob_4 >= 0.5).astype(int) # threshold 0.5

print("Accuracy:", accuracy_score(y_test, y_pred_4))
print("Precision:", precision_score(y_test, y_pred_4))
print("Recall:", recall_score(y_test, y_pred_4))
print("F1-score:", f1_score(y_test, y_pred_4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_4))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.7877927608232789
Precision: 0.6169491525423729
Recall: 0.4945652173913043
F1-score: 0.5490196078431373
ROC-AUC: 0.8217276448231216


## Model 5

In [21]:
early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    mode="max",
    restore_best_weights=True
)

In [22]:
model = build_model([128, 64, 32, 16], dropout_rate=0.2)

history = model.fit(
    X_train_processed,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7386 - auc: 0.7392 - loss: 0.5136 - val_accuracy: 0.8234 - val_auc: 0.8489 - val_loss: 0.4077
Epoch 2/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7843 - auc: 0.8229 - loss: 0.4471 - val_accuracy: 0.8154 - val_auc: 0.8522 - val_loss: 0.4066
Epoch 3/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7910 - auc: 0.8314 - loss: 0.4388 - val_accuracy: 0.8146 - val_auc: 0.8541 - val_loss: 0.4035
Epoch 4/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7979 - auc: 0.8373 - loss: 0.4322 - val_accuracy: 0.8083 - val_auc: 0.8533 - val_loss: 0.4068
Epoch 5/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8023 - auc: 0.8385 - loss: 0.4310 - val_accuracy: 0.8128 - val_auc: 0.8536 - val_loss: 0.4042
Epoch 6/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8063 - auc: 0.8407 - loss: 0.4284 - val_accuracy: 0.8119 - val_auc: 0.8536 - val_loss: 0.4043
Epoch 7/50
141/141 ━━━━━━━━━━━━━━━━━━━━ 

### Model 5- Evaluation

In [23]:
y_prob_5 = model.predict(X_test_processed).ravel()

y_pred_5 = (y_prob_5 >= 0.5).astype(int) # threshold 0.5

print("Accuracy:", accuracy_score(y_test, y_pred_5))
print("Precision:", precision_score(y_test, y_pred_5))
print("Recall:", recall_score(y_test, y_pred_5))
print("F1-score:", f1_score(y_test, y_pred_5))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_5))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Accuracy: 0.7849538679914834
Precision: 0.6406926406926406
Recall: 0.40217391304347827
F1-score: 0.4941569282136895
ROC-AUC: 0.8261835400743431


## Neural Network Architecture Comparison

In [24]:
nn_comparison = pd.DataFrame({
    "Architecture": [
        "32 → 16",
        "16 → 8",
        "64 → 32",
        "64 → 32 → 16",
        "128 → 64 → 32 → 16"
    ],
    "Accuracy": [
        0.7857,
        0.7892,
        0.7857,
        0.7878,
        0.7850
    ],
    "Precision": [
        0.5965,
        0.6127,
        0.6204,
        0.6169,
        0.6407
    ],
    "Recall": [
        0.5543,
        0.5245,
        0.4620,
        0.4946,
        0.4022
    ],
    "F1-score": [
        0.5746,
        0.5652,
        0.5296,
        0.5490,
        0.4942
    ],
    "ROC-AUC": [
        0.8188,
        0.8184,
        0.8237,
        0.8217,
        0.8262
    ]
})

nn_comparison.round(4)

,Architecture,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,32 → 16,0.7857,0.5965,0.5543,0.5746,0.8188
1,16 → 8,0.7892,0.6127,0.5245,0.5652,0.8184
2,64 → 32,0.7857,0.6204,0.4620,0.5296,0.8237
3,64 → 32 → 16,0.7878,0.6169,0.4946,0.5490,0.8217
4,128 → 64 → 32 → 16,0.7850,0.6407,0.4022,0.4942,0.8262


## Neural Network Architecture Comparison

Five neural network architectures were evaluated using the same preprocessing pipeline and training procedure. The architectures produced similar ROC-AUC values, with no substantial improvement from increasing the network size or depth.

The `32 → 16` architecture was selected as the final neural network model because it provided the best balance between Recall and F1-score, while maintaining competitive Accuracy and ROC-AUC. Larger architectures achieved slightly higher ROC-AUC in some cases, but this did not translate into better performance at the default classification threshold of 0.5.

## Conclusion

The neural network experiments showed that increasing model complexity did not provide a meaningful improvement for this dataset. The selected `32 → 16` architecture achieved a ROC-AUC of approximately 0.82, but did not outperform the Logistic Regression or XGBoost models evaluated earlier.

This suggests that the Telco Customer Churn dataset can be modeled effectively without a particularly deep neural network. In this case, model complexity alone was not sufficient to improve predictive performance.

The neural network was therefore retained as an additional model for comparison rather than as the final best-performing model. The results also highlight the importance of evaluating multiple metrics: a model with a slightly higher ROC-AUC does not necessarily provide better Recall or F1-score at a specific classification threshold.
